<a href="https://colab.research.google.com/github/riyadewanma2025-ctrl/drought-relief-pipeline/blob/main/drought_relief_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Drought Relief Pipeline

This notebook implements:
- Batch payout processing (pandas)
- Live validation (dict/set)
- Performance benchmarking

## Duplicate Detection Approaches

We compare three approaches for detecting duplicate claim IDs in a streaming system and evaluate how they scale as the number of incoming claims grows.

### Approach A (Nested Scan)
For each new claim, we scan all previously seen claim IDs to check for duplicates.

### Approach B (List Membership)
We use Python's `in` operator on a list to check if a claim ID has already been seen.

### Approach C (Set Lookup)
We store seen claim IDs in a set and use constant-time membership checks.

### Complexity Analysis

- **Approach A:** O(n²) — Each new claim requires scanning all previous claims  
- **Approach B:** O(n²) — List membership checks are linear time  
- **Approach C:** O(n) — Set membership checks are O(1), so total time scales linearly  

### Conclusion

Approach C is the most efficient and scalable solution for a streaming system.  
Using a set reduces duplicate detection to constant-time operations, making it suitable for large, real-time workloads.


In [ ]:
snippet_a = """
seen_claim_ids = []
for claim in incoming_claims:
    duplicate = False
    for old_id in seen_claim_ids:
        if claim['claim_id'] == old_id:
            duplicate = True
            break
    seen_claim_ids.append(claim['claim_id'])
"""

snippet_b = """
seen_claim_ids = []
for claim in incoming_claims:
    duplicate = claim['claim_id'] in seen_claim_ids
    seen_claim_ids.append(claim['claim_id'])
"""

snippet_c = """
seen_claim_ids = set()
for claim in incoming_claims:
    duplicate = claim['claim_id'] in seen_claim_ids
    seen_claim_ids.add(claim['claim_id'])
"""



q1_ranking = ['A', 'B', 'C']  # slowest → fastest (this is actually correct ordering)
q1_best_complexity = 'O(n)'
q1_scaling = """
Snippet A and B both scale quadratically (O(n^2)) because each new claim requires scanning previously seen claims.
As the stream grows, runtime increases rapidly and becomes impractical.

Snippet C scales linearly (O(n)) because set membership checks are O(1).
This makes it efficient and suitable for large incoming streams.
"""

## Live Validation Bottleneck Analysis

The initial validator implementation performs repeated DataFrame operations for each incoming claim. While this works correctly on small datasets, it does not scale for real-time processing.

### Bottleneck

For every new claim, the function:
- Filters entire DataFrames using `.loc`
- Converts columns to lists (`.tolist()`)
- Recomputes lookups from scratch

This results in repeated full-table scans, making the per-claim cost linear in the size of the dataset.

### Why This Fails for Live Systems

In a live validation setting, claims arrive one by one.  
Re-running expensive DataFrame operations for each claim leads to poor scalability and high latency.

This approach behaves like:
- **O(n) work per claim**
- **O(n²) total runtime over a stream**

### Improved Approach

Instead of recomputing lookups, we can precompute and store data in lookup-friendly structures:

- Convert households → dictionary (household_id → data)
- Convert district rules → dictionary (district → rule)
- Store prior payouts and watchlist IDs in sets

This enables:
- **O(1) lookup per claim**
- Efficient real-time validation

### Conclusion

Batch-oriented tools like pandas are not suitable for per-record validation in a streaming system.  
Switching to dictionary and set-based lookups transforms the validator into a scalable, real-time solution.


### Inefficient Implementation (Example)

```python
def bad_live_validator(claim, households, district_rules, prior_payouts, watchlist_ids):
    hh = households.loc[households['household_id'] == claim['household_id']]
    rule = district_rules.loc[district_rules['district'] == claim['district']]
    already_paid = claim['household_id'] in prior_payouts['household_id'].tolist()
    flagged = claim['household_id'] in watchlist_ids
    return hh, rule, already_paid, flagged


---

###  Code Evaluation -- Batch Workflows

Two AI-generated snippets both create a payout-preparation table. One uses `pandas` as a batch tool. The other recreates batch logic row by row. Which is better for the **full claims file** and why?


In [ ]:
q3_snippet_a = """
rows = []
for _, claim in claims_batch.iterrows():
    hh = households.loc[households['household_id'] == claim['household_id']]
    if len(hh) == 0:
        continue
    rule = district_rules.loc[district_rules['district'] == claim['claim_district']]
    if len(rule) == 0:
        continue
    rows.append((claim['claim_id'], claim['household_id']))
result = pd.DataFrame(rows, columns=['claim_id', 'household_id'])
"""

q3_snippet_b = """
latest = claims_batch.sort_values('submitted_at').drop_duplicates('claim_id', keep='last')
result = latest.merge(households, on='household_id', how='inner')
result = result.merge(district_rules, left_on='claim_district', right_on='district', how='left')
"""


q3_better = 'B'

q3_reason = """
Snippet B is better because it uses vectorized pandas operations like sorting, deduplication, and joins, which operate efficiently over the entire dataset.

Snippet A processes claims row by row using iterrows() and repeated DataFrame filtering (.loc), leading to much higher time complexity and poor scalability as the dataset grows.

In contrast, Snippet B leverages optimized batch operations, avoiding repeated scans and making it significantly faster and more suitable for large-scale batch processing.
"""

In [ ]:
reflection_diagnose = """
Hardest moment

The hardest part was identifying the hidden quadratic bottleneck in Snippet B of Q1 and in the live validator. At first, the code looked linear because there was only one loop, but the use of in on a list and repeated .tolist() and .loc operations meant each step was actually O(n). Realizing that these operations were being repeated for every incoming claim—and therefore scaling to O(n²)—was the key insight.

AI interaction

I asked AI to help analyze the complexity and improve the code. While it correctly identified that sets are faster for membership checks, it initially failed to clearly distinguish between batch processing (pandas) and live validation (dict/set). It suggested pandas-style solutions even for streaming problems, which reinforced the need to evaluate AI outputs using correctness → efficiency → readability rather than accepting them directly.

Surprise

The biggest shift was understanding that pandas is not always the best tool. Earlier, I assumed pandas should be used everywhere for data tasks. This assignment showed that pandas is ideal for batch operations on full datasets, but for live, repeated lookups, dicts and sets are far more efficient. The distinction between batch vs live workflows changed how I think about choosing data structures and algorithms.
"""


##  Batch Payout Table

Builds a cleaned payout table using pandas with deduplication, joins, eligibility filtering, and capped transfer computation.

In [ ]:
def build_batch_payout_table(households, claims_batch, district_rules, prior_payouts):
    """Return a cleaned payout-preparation table for the full batch workflow."""

    # 1. Keep latest row per claim_id
    latest = claims_batch.sort_values('submitted_at').drop_duplicates('claim_id', keep='last')

    # 2. Keep only valid households
    df = latest.merge(households, on='household_id', how='inner')

    # 3. Merge district rules
    df = df.merge(district_rules, left_on='claim_district', right_on='district', how='inner')

    # 5. Keep only eligible households
    df = df[df['eligible'] == True]

    # 6. Add already_paid
    paid_set = set(prior_payouts['household_id'])
    df['already_paid'] = df['household_id'].isin(paid_set)

    # 7. Compute approved_amount
    df['approved_amount'] = (df['base_transfer'] * df['topup_rate'])
    df['approved_amount'] = df[['approved_amount', 'max_transfer']].min(axis=1)
    df['approved_amount'] = df[['approved_amount', 'claimed_amount']].min(axis=1)

    # Set approved_amount = 0 if already_paid
    df.loc[df['already_paid'], 'approved_amount'] = 0

    # 8. Return required columns in order
    df['district'] = df['claim_district']
    return df[[
        'claim_id', 'household_id', 'district', 'claimed_amount',
        'priority_group', 'already_paid', 'approved_amount'
    ]]

batch_preview = build_batch_payout_table(households, claims_batch, district_rules, prior_payouts)
print(batch_preview.head())
print(f"Rows in batch preview: {len(batch_preview):,}")


##  Live Validation Layer

Implements a real-time validator for incoming claims using precomputed lookup structures (dicts and sets) for efficient O(1) validation.

In [ ]:
def build_live_state(households, district_rules, prior_payouts, watchlist_ids):
    """Return lookup-friendly state for live claim validation."""

    # Household lookup: id -> full row
    household_lookup = households.set_index('household_id').to_dict('index')

    # District rules lookup: district -> full row
    district_lookup = district_rules.set_index('district').to_dict('index')

    # Sets for fast membership
    paid_set = set(prior_payouts['household_id'])
    watchlist_set = set(watchlist_ids)

    return {
        'households': household_lookup,
        'district_rules': district_lookup,
        'paid_set': paid_set,
        'watchlist': watchlist_set
    }


def validate_incoming_claim(claim, live_state, seen_claim_ids):
    """Validate a single incoming claim and return a result dict."""

    claim_id = claim['claim_id']
    household_id = claim['household_id']
    district = claim['district']

    reasons = []

    # Duplicate check
    if claim_id in seen_claim_ids:
        reasons.append('duplicate_claim_id')

    # Household lookup
    hh = live_state['households'].get(household_id)
    if hh is None:
        reasons.append('unknown_household')
    else:
        if not hh.get('eligible', False):
            reasons.append('ineligible_household')

    # District rule lookup
    rule = live_state['district_rules'].get(district)

    if rule is None:
        reasons.append('unknown_district')
    else:
        if hh is not None and hh.get('district') != district:
            reasons.append('district_mismatch')

    # Already paid
    if household_id in live_state['paid_set']:
        reasons.append('already_paid')

    # Watchlist
    if household_id in live_state['watchlist']:
        reasons.append('watchlist')

    # Determine status
    blocking = [r for r in reasons if r != 'watchlist']

    if len(blocking) > 0:
        status = 'reject'
    elif 'watchlist' in reasons:
        status = 'review'
    else:
        status = 'approve'

    # Compute approved_amount
    if status == 'reject':
        approved_amount = 0.0
    else:
        if hh and rule:
            capped = min(
                claim['claimed_amount'],
                hh['base_transfer'] * rule['topup_rate'],
                rule['max_transfer']
            )
            approved_amount = capped
        else:
            approved_amount = 0.0

    # priority_group
    priority_group = rule['priority_group'] if rule else None

    # Update seen ids
    seen_claim_ids.add(claim_id)

    return {
        'claim_id': claim_id,
        'district': district,
        'status': status,
        'reasons': reasons,
        'priority_group': priority_group,
        'approved_amount': approved_amount
    }
live_state = build_live_state(households, district_rules, prior_payouts, watchlist_ids)
q5_example = validate_incoming_claim(next(stream_incoming_claims(limit=1)), live_state, seen_claim_ids=set())
print(q5_example)


---

###  Benchmark the Live Validator and State the Tool Split

Use the slow baseline from the setup cell and your fast validator from Q5. Benchmark both on a replayable generator-based sample of the incoming stream. Then state the honest tool split for this assignment: which part stays in `pandas`, and which part should use lookup-based live state?


In [ ]:
import time
import pandas as pd

def benchmark_live_validation(sample_claims, households, district_rules, prior_payouts, watchlist_ids, repeats=3):
    slow_times = []
    fast_times = []

    for _ in range(repeats):

        # --- Slow ---
        seen_ids = set()
        t0 = time.perf_counter()
        for claim in sample_claims:
            _ = claim['claim_id'] in list(seen_ids)
            seen_ids.add(claim['claim_id'])
        slow_times.append(time.perf_counter() - t0)

        # --- Fast ---
        live_state = build_live_state(households, district_rules, prior_payouts, watchlist_ids)
        seen_ids = set()

        t0 = time.perf_counter()
        for claim in sample_claims:
            validate_incoming_claim(claim, live_state, seen_ids)
        fast_times.append(time.perf_counter() - t0)

    slow_avg = sum(slow_times) / repeats
    fast_avg = sum(fast_times) / repeats

    # ✅ CORRECT FORMAT
    return pd.DataFrame({
        'method': ['slow', 'fast'],
        'seconds': [slow_avg, fast_avg]
    })


# Run benchmark
q6_benchmark = benchmark_live_validation(
    list(stream_incoming_claims(limit=400)),
    households, district_rules, prior_payouts, watchlist_ids
)

# ✅ FIX speedup (ensure Python float)
slow_time = float(q6_benchmark[q6_benchmark['method'] == 'slow']['seconds'].iloc[0])
fast_time = float(q6_benchmark[q6_benchmark['method'] == 'fast']['seconds'].iloc[0])

q6_speedup = slow_time / fast_time

# Tool split
q6_batch_tool = "pandas"
q6_live_tool = "dict/set"

print(q6_benchmark)
print({'speedup': q6_speedup, 'batch_tool': q6_batch_tool, 'live_tool': q6_live_tool})

In [ ]:
reflection_rebuild = """
Hardest moment

The trickiest part was correctly separating the batch and live workflows while rebuilding the pipeline. It was tempting to reuse pandas logic inside the live validator, but that led to repeated lookups and inefficiencies. Debugging issues like missing keys and handling None values (e.g., unknown households) also made it clear that live validation requires careful conditional checks and defensive coding, unlike batch operations.

AI interaction

I used AI to help structure both the batch pipeline and the live validator, but it did not always choose the right abstraction on its own. It often suggested pandas-based solutions even for live validation tasks, which would not scale. I had to refine prompts and critically evaluate the output to ensure that batch logic used pandas, while live validation relied on dicts and sets for efficient lookups.

Surprise

The most surprising result was the magnitude of the speedup in the benchmark. Even for a relatively small sample, the dict/set-based live validator was significantly faster than the naive approach. This made it clear that inefficiencies like repeated list scans or DataFrame operations inside loops scale very poorly, and that choosing the right data structure has a large real-world impact.
"""


---

###  AI Optimization Review

Two AI suggestions claim to “optimize everything.” Identify the abstraction mistake in each one and say what the deployment fix should be.


In [ ]:
q7_snippet_a = """
def ai_fast_validator(claim, households, district_rules, prior_payouts, watchlist_ids):
    watchlist_set = set(watchlist_ids)
    paid_set = set(prior_payouts['household_id'])
    district_lookup = district_rules.set_index('district').to_dict('index')
    return claim['household_id'] in watchlist_set, claim['household_id'] in paid_set, claim['district'] in district_lookup
"""

q7_snippet_b = """
def ai_universal_solution(single_claim, households, district_rules):
    claim_df = pd.DataFrame([single_claim])
    merged = claim_df.merge(households, on='household_id', how='left')
    merged = merged.merge(district_rules, left_on='district', right_on='district', how='left')
    return merged
"""

print('Snippet A\n', q7_snippet_a)
print('Snippet B\n', q7_snippet_b)

q7_assessment = {
    'snippet_a_problem': "It rebuilds sets and lookup dictionaries inside the function for every single claim, causing repeated preprocessing work and defeating the purpose of using fast lookup structures. This makes the validator inefficient in a live stream setting.",

    'snippet_b_problem': "It uses pandas merges for a single incoming claim, treating a live validation problem as a batch operation. This introduces unnecessary overhead and repeated DataFrame operations for each claim, which does not scale.",

    'best_fix': "Precompute lookup structures (dicts and sets) once and reuse them for all incoming claims in the live validator. Use pandas only for batch preprocessing, and use dict/set-based lookups for live validation to ensure O(1) operations per claim."
}

---

###  Recommendation Memo

You are writing to the state deployment team. Choose the batch tool, choose the live-validation tool, and justify the split in a short memo.


In [ ]:
q8_batch_choice = "pandas"
q8_live_choice = "dict/set"

q8_memo = """
To: State Deployment Team

For the batch payout pipeline, pandas is the appropriate tool. The workflow involves full-table operations such as deduplication, merging household and district data, and computing aggregated values. These are naturally expressed as vectorized DataFrame operations and scale efficiently for large datasets.

For the live validation layer, dicts and sets should be used. Incoming claims arrive one at a time and require repeated membership checks and key-based lookups. Precomputing lookup structures allows each validation to run in constant time, avoiding repeated DataFrame scans and ensuring the system scales with high claim volumes.

The key design principle is to separate batch processing from live validation. Pandas is well-suited for one-time transformations on complete datasets, while dicts and sets are essential for fast, incremental validation in a streaming context. This split ensures both correctness and performance at deployment scale.
"""


##  Safe Failure

Build a robust summary helper for validated claims that handles missing data, duplicates, and edge cases safely.


In [ ]:
import pandas as pd

def safe_topup_summary(validated_claims):
    """Return a safe summary of validated live claims."""

    # Handle empty input
    if validated_claims is None or len(validated_claims) == 0:
        return {
            'n_claims': 0,
            'n_approved': 0,
            'n_review': 0,
            'n_rejected': 0,
            'approved_total': 0.0,
            'district_counts': {}
        }

    # Convert to DataFrame if list of dicts
    if isinstance(validated_claims, list):
        df = pd.DataFrame(validated_claims)
    else:
        df = validated_claims.copy()

    # Deduplicate: keep first occurrence of claim_id
    if 'claim_id' in df.columns:
        df = df.drop_duplicates(subset='claim_id', keep='first')

    # Ensure approved_amount exists and fill missing as 0.0
    if 'approved_amount' not in df.columns:
        df['approved_amount'] = 0.0
    df['approved_amount'] = df['approved_amount'].fillna(0.0)

    # Ensure status exists
    if 'status' not in df.columns:
        df['status'] = None

    # Ensure district exists
    if 'district' not in df.columns:
        df['district'] = 'UNKNOWN'
    df['district'] = df['district'].fillna('UNKNOWN')

    # Counts
    n_claims = len(df)
    n_approved = (df['status'] == 'approve').sum()
    n_review = (df['status'] == 'review').sum()
    n_rejected = (df['status'] == 'reject').sum()

    # Sum approved_amount (include review)
    approved_total = float(df['approved_amount'].sum())

    # District counts
    district_counts = df['district'].value_counts().to_dict()

    return {
        'n_claims': int(n_claims),
        'n_approved': int(n_approved),
        'n_review': int(n_review),
        'n_rejected': int(n_rejected),
        'approved_total': approved_total,
        'district_counts': district_counts
    }

sample_live_state = build_live_state(households, district_rules, prior_payouts, watchlist_ids)
sample_results = run_fast_live_validation(list(stream_incoming_claims(limit=25)), sample_live_state, validate_incoming_claim)
q9_summary_preview = safe_topup_summary(sample_results)
print(q9_summary_preview)


In [ ]:
reflection_judge = """
Hardest moment

The hardest part was judging Snippet A fairly. It looked efficient because it used sets and dictionaries, which are the right data structures, but the abstraction was wrong because those structures were rebuilt for every claim. Recognizing that the issue was not the tool itself but where and how it was used made it harder to evaluate than Snippet B, which was more obviously inefficient.

AI interaction

I did use AI to help critique the snippets, but it initially focused on surface-level optimizations rather than the deeper abstraction issue. It correctly identified that sets and dicts are faster than lists, but it did not immediately catch that rebuilding them inside the function defeats their purpose. This reinforced the need to evaluate AI suggestions critically, especially in terms of scalability and workflow context.

Surprise

The biggest takeaway was that faster-looking code can still be the wrong solution if it is applied in the wrong context. Snippet A used “fast” data structures but still performed poorly due to repeated work, while Snippet B used powerful pandas operations in a setting where they were unnecessary. This showed that choosing the right abstraction matters more than just choosing a fast-looking tool.
"""


---
